# One-vs-Rest (OvR) — Pig Posture Classification

5 binary EfficientNetV2-S classifiers, one per class.  
Disagreements resolved by a calibrated meta-classifier.

| | |
|---|---|
| **Backbone** | EfficientNetV2-S (ImageNet) |
| **Loss** | BCEWithLogitsLoss + pos_weight |
| **Sampler** | WeightedRandomSampler per binary task |
| **Fine-tuning** | 2-stage: head → full model |
| **Agreement** | threshold=0.5 per classifier |
| **Tiebreaker** | LogReg + isotonic calibration on (N×5) prob matrix |

In [ ]:
!pip install albumentations -q

# ── Colab setup (skip if running locally) ────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import files
    files.upload()   # kaggle.json
    import os, shutil
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    !kaggle competitions download -c multi-view-pig-posture-recognition
    !unzip -q multi-view-pig-posture-recognition.zip

In [ ]:
import ast, copy, os, pickle, re, time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
import torchvision.transforms.functional as TF

from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
CLASS_NAMES = {
    0: 'Lateral_lying_left',
    1: 'Lateral_lying_right',
    2: 'Sitting',
    3: 'Standing',
    4: 'Sternal_lying',
}
NUM_CLASSES = len(CLASS_NAMES)

BASE_DIR    = Path('multiview_pig_posture_recognition')
TRAIN2_IMGS = BASE_DIR / 'train2_images'
TEST_IMGS   = BASE_DIR / 'test_images'

BATCH_SIZE    = 32
OVR_THRESHOLD = 0.5   # per-classifier yes/no cutoff
META_VAL_SIZE = 0.20  # fraction of train2 held out for meta-clf training

# Fine-tuning schedule per binary model
EPOCHS_HEAD = 3;  LR_HEAD = 1e-3
EPOCHS_FULL = 8;  LR_FULL = 1e-4

# Save paths
OVR_MODEL_DIR = Path('ovr_models')
OVR_MODEL_DIR.mkdir(exist_ok=True)
META_CLF_PATH = OVR_MODEL_DIR / 'meta_classifier.pkl'

def ovr_path(class_id):
    return OVR_MODEL_DIR / f'ovr_class{class_id}_{CLASS_NAMES[class_id]}.pt'

print('Config OK')
for i, name in CLASS_NAMES.items():
    print(f'  [{i}] {name:25s} -> {ovr_path(i).name}')

## Data loading & camera augmentations

In [ ]:
def parse_camera_meta(image_id):
    m = re.match(r'(pen\d+)_(orb|tur)_(cam\d+)_', str(image_id))
    return (m.group(1), m.group(2), m.group(3)) if m else ('unknown','unknown','unknown')

def add_camera_cols(df):
    df['pen']      = df['image_id'].apply(lambda x: parse_camera_meta(x)[0])
    df['cam_type'] = df['image_id'].apply(lambda x: parse_camera_meta(x)[1])
    df['cam_num']  = df['image_id'].apply(lambda x: parse_camera_meta(x)[2])
    df['camera']   = df['pen'] + '_' + df['cam_type'] + '_' + df['cam_num']
    return df

train2 = pd.read_csv(BASE_DIR / 'train2.csv')
train2['source']      = 'train2'
train2['bbox_parsed'] = train2['bbox'].apply(ast.literal_eval)
train2 = add_camera_cols(train2)

test = pd.read_csv(BASE_DIR / 'test.csv')
test['source']      = 'test'
test['bbox_parsed'] = test['bbox'].apply(ast.literal_eval)
test = add_camera_cols(test)

# Stratified split: 80% for binary model training, 20% for meta-clf training
train_df, val_df = train_test_split(
    train2, test_size=META_VAL_SIZE,
    stratify=train2['class_id'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f'Train2 total:  {len(train2):,}')
print(f'Binary train:  {len(train_df):,}')
print(f'Meta val:      {len(val_df):,}')
print('\nClass distribution (val):')
for cid, cnt in val_df['class_id'].value_counts().sort_index().items():
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt}')

In [ ]:
# Camera augmentation functions
# Horizontal-flip cameras require label swap: Lateral_lying_left(0) <-> Lateral_lying_right(1)
FLIP_LABEL_MAP = {0: 1, 1: 0, 2: 2, 3: 3, 4: 4}

def aug_pen2_tur_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_saturation(img, 0.8)
    return TF.adjust_brightness(img, 0.95)

def aug_pen1_tur_cam2(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, 1.35)
    return TF.adjust_saturation(img, 0.9)

def aug_pen2_orb_cam1(img):
    img = TF.hflip(img)
    img = TF.adjust_brightness(img, 0.6)
    img = TF.adjust_contrast(img, 1.5)
    arr = np.array(img).astype(float)
    arr[:,:,0] = (arr[:,:,0]*0.85).clip(0,255)
    arr[:,:,1] = (arr[:,:,1]*1.10).clip(0,255)
    arr[:,:,2] = (arr[:,:,2]*0.80).clip(0,255)
    shifted = Image.fromarray(arr.clip(0,255).astype(np.uint8))
    grey    = np.array(TF.to_grayscale(shifted, 3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

def aug_pen2_tur_cam2(img):
    img = TF.adjust_brightness(img, 1.3)
    return TF.adjust_saturation(img, 0.85)

def aug_pen2_orb_cam2(img):
    img  = TF.adjust_brightness(img, 0.60)
    img  = TF.adjust_contrast(img, 1.5)
    arr  = np.array(img).astype(float)
    grey = np.array(TF.to_grayscale(img, 3)).astype(float)
    return Image.fromarray((0.65*arr + 0.35*grey).clip(0,255).astype(np.uint8))

# (aug_fn, does_horizontal_flip)
CAMERA_AUG_FN = {
    'pen2_tur_cam1': (aug_pen2_tur_cam1, True),
    'pen1_tur_cam2': (aug_pen1_tur_cam2, True),
    'pen2_orb_cam1': (aug_pen2_orb_cam1, True),
    'pen2_tur_cam2': (aug_pen2_tur_cam2, False),
    'pen2_orb_cam2': (aug_pen2_orb_cam2, False),
}

print('Camera augmentations:')
for cam, (_, flip) in CAMERA_AUG_FN.items():
    print(f'  {cam:20s}  flip={flip}')

## Transforms & datasets

In [ ]:
MU  = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

class AddGaussianNoise:
    def __init__(self, std=0.02, p=0.15):
        self.std, self.p = std, p
    def __call__(self, t):
        if torch.rand(1).item() < self.p:
            t = torch.clamp(t + torch.randn_like(t)*self.std, 0., 1.)
        return t

TRAIN_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    AddGaussianNoise(0.02, 0.15),
    transforms.Normalize(MU, STD),
])
VAL_TF = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(MU, STD),
])


def load_image(image_id, source):
    folder = {'train2': TRAIN2_IMGS, 'test': TEST_IMGS}[source]
    return Image.open(folder / image_id).convert('RGB')

def crop_with_padding(image, bbox, padding=0.12):
    img_w, img_h = image.size
    x, y, w, h   = map(float, bbox)
    x1 = x - w*padding;      y1 = y - h*padding
    x2 = x + w + w*padding;  y2 = y + h + h*padding
    side   = max(x2-x1, y2-y1)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    x1, x2 = cx-side/2, cx+side/2
    y1, y2 = cy-side/2, cy+side/2
    x1 = max(0, int(round(x1)));      y1 = max(0, int(round(y1)))
    x2 = min(img_w, int(round(x2)));  y2 = min(img_h, int(round(y2)))
    return image.crop((x1, y1, max(x2,x1+1), max(y2,y1+1)))


class BinaryDataset(Dataset):
    """
    One-vs-Rest dataset for a single target_class.
    Label: 1.0 if instance belongs to target_class, else 0.0.
    Camera augmentations applied when is_train=True,
    with correct label remapping for horizontal-flip cameras.
    """
    def __init__(self, df, target_class, transform, is_train=True):
        self.transform    = transform
        self.target_class = target_class
        df = df.reset_index(drop=True)
        self.df = df
        self.samples = []  # (row_idx, do_aug, binary_label)

        for idx, row in df.iterrows():
            cam        = row.get('camera', 'unknown')
            orig_label = int(row['class_id'])
            bin_label  = float(orig_label == target_class)
            self.samples.append((idx, False, bin_label))

            if is_train and cam in CAMERA_AUG_FN:
                _, does_flip = CAMERA_AUG_FN[cam]
                aug_label    = FLIP_LABEL_MAP[orig_label] if does_flip else orig_label
                self.samples.append((idx, True, float(aug_label == target_class)))

    def __len__(self):  return len(self.samples)

    def __getitem__(self, i):
        row_i, do_aug, bin_label = self.samples[i]
        row  = self.df.iloc[row_i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        if do_aug:
            aug_fn, _ = CAMERA_AUG_FN[row['camera']]
            crop = aug_fn(crop)
        return self.transform(crop), torch.tensor(bin_label)


class InferenceDataset(Dataset):
    """Label-free dataset for inference."""
    def __init__(self, df, transform):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):  return len(self.df)
    def __getitem__(self, i):
        row  = self.df.iloc[i]
        img  = load_image(row['image_id'], row['source'])
        crop = crop_with_padding(img, row['bbox_parsed'])
        return self.transform(crop), str(row.get('row_id', row['image_id']))


print('Datasets OK')

## Model & training utilities

In [ ]:
def create_binary_model():
    """EfficientNetV2-S with a single sigmoid output for binary classification."""
    m = models.efficientnet_v2_s(
        weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 1)
    return m.to(DEVICE)

def freeze_backbone(model):
    for p in model.features.parameters():   p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True

def unfreeze_all(model):
    for p in model.parameters(): p.requires_grad = True

def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def make_binary_loader(df, target_class, is_train):
    tf  = TRAIN_TF if is_train else VAL_TF
    ds  = BinaryDataset(df, target_class, tf, is_train=is_train)
    labels = np.array([s[2] for s in ds.samples])
    if is_train:
        counts  = np.bincount((labels > 0.5).astype(int), minlength=2)
        w       = (1.0 / np.maximum(counts, 1))[(labels > 0.5).astype(int)]
        sampler = WeightedRandomSampler(torch.DoubleTensor(w), len(w), replacement=True)
        return DataLoader(ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=2, pin_memory=True)


def train_one_epoch_binary(model, loader, optimizer, criterion):
    model.train()
    total_loss, all_preds, all_targets = 0., [], []
    n = len(loader)
    for bi, (imgs, labels) in enumerate(loader):
        imgs   = imgs.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(out) > OVR_THRESHOLD).squeeze(1).long()
        all_preds.extend(preds.detach().cpu().tolist())
        all_targets.extend(labels.squeeze(1).long().cpu().tolist())
        bar = '\u2588'*int(30*(bi+1)/n) + '\u2591'*(30-int(30*(bi+1)/n))
        print(f'\r  |{bar}| {100*(bi+1)/n:5.1f}%  loss={loss.item():.4f}',
              end='', flush=True)
    print()
    f1 = f1_score(all_targets, all_preds, average='binary', zero_division=0)
    return total_loss / len(loader.dataset), f1


print('Model utils OK')

In [ ]:
def train_binary_model(target_class, train_df, save_path=None):
    """
    Train a single OvR binary classifier for `target_class`.
    Returns (model, best_f1).
    """
    class_name = CLASS_NAMES[target_class]
    n_pos      = (train_df['class_id'] == target_class).sum()
    n_neg      = len(train_df) - n_pos
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32).to(DEVICE)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    print(f'\n{"="*65}')
    print(f'OvR  Class {target_class}: {class_name}')
    print(f'  Positive: {n_pos:,}  |  Negative: {n_neg:,}  |  pos_weight={pos_weight.item():.2f}')
    print(f'{"="*65}')

    model      = create_binary_model()
    best_f1    = 0.
    best_state = None

    # Stage 1 — head only
    freeze_backbone(model)
    loader = make_binary_loader(train_df, target_class, is_train=True)
    opt    = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR_HEAD
    )
    print(f'\n  Stage 1 (head only, {n_trainable(model):,} params)')
    for ep in range(EPOCHS_HEAD):
        t0 = time.time()
        loss, f1 = train_one_epoch_binary(model, loader, opt, criterion)
        flag = ' <- best' if f1 > best_f1 else ''
        print(f'  Ep {ep+1}/{EPOCHS_HEAD}  loss={loss:.4f}  f1={f1:.4f}  '
              f'({(time.time()-t0)/60:.1f}m){flag}')
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())

    # Stage 2 — full model, cosine LR
    unfreeze_all(model)
    loader = make_binary_loader(train_df, target_class, is_train=True)
    opt    = torch.optim.Adam(model.parameters(), lr=LR_FULL)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FULL, eta_min=1e-6)
    print(f'\n  Stage 2 (full, {n_trainable(model):,} params)')
    for ep in range(EPOCHS_FULL):
        t0 = time.time()
        loss, f1 = train_one_epoch_binary(model, loader, opt, criterion)
        sched.step()
        flag = ' <- best' if f1 > best_f1 else ''
        print(f'  Ep {ep+1}/{EPOCHS_FULL}  loss={loss:.4f}  f1={f1:.4f}  '
              f'lr={opt.param_groups[0]["lr"]:.1e}  ({(time.time()-t0)/60:.1f}m){flag}')
        if f1 > best_f1:
            best_f1    = f1
            best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    if save_path:
        torch.save({
            'model_state_dict': best_state,
            'target_class':     target_class,
            'class_name':       class_name,
            'best_train_f1':    best_f1,
            'pos_weight':       pos_weight.item(),
            'ovr_threshold':    OVR_THRESHOLD,
        }, save_path)
        print(f'\n  Saved -> {save_path}  (best F1={best_f1:.4f})')

    return model, best_f1


print('train_binary_model() OK')

## Train all 5 binary classifiers

**Skip this cell and run the *Load models* cell below if you already have saved `.pt` files.**

In [ ]:
ovr_models    = {}
ovr_train_f1s = {}

for class_id in range(NUM_CLASSES):
    model_i, f1_i = train_binary_model(
        target_class=class_id,
        train_df=train_df,
        save_path=ovr_path(class_id),
    )
    ovr_models[class_id]    = model_i
    ovr_train_f1s[class_id] = f1_i

print('\n' + '='*65)
print('OvR training summary:')
for cid, f1 in ovr_train_f1s.items():
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: F1={f1:.4f}')

## Load individual models from file

Run this instead of (or after) the training cell to load saved models.  
You can load all 5 at once, or specific classes only (`class_ids=[0, 2]`).

In [ ]:
def load_binary_models(model_dir=OVR_MODEL_DIR, class_ids=None):
    """
    Load OvR binary models from disk.

    Args:
        model_dir:  directory containing ovr_classX_*.pt files
        class_ids:  list of class indices to load; None = load all 5

    Returns:
        dict {class_id: model}  (models are in eval mode on DEVICE)
    """
    if class_ids is None:
        class_ids = list(range(NUM_CLASSES))

    loaded = {}
    for cid in class_ids:
        path = Path(model_dir) / f'ovr_class{cid}_{CLASS_NAMES[cid]}.pt'
        if not path.exists():
            print(f'  [{cid}] FILE NOT FOUND: {path}')
            continue
        ckpt  = torch.load(path, map_location=DEVICE)
        model = create_binary_model()
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        model.to(DEVICE)
        loaded[cid] = model
        print(f'  [{cid}] {CLASS_NAMES[cid]:25s}  '
              f'F1={ckpt.get("best_train_f1", 0.):.4f}  '
              f'pos_weight={ckpt.get("pos_weight", "?"):.2f}  <- loaded')

    missing = set(range(NUM_CLASSES)) - set(loaded.keys())
    if missing:
        print(f'\nWARNING: missing models for classes {sorted(missing)}')
    else:
        print('\nAll 5 models loaded OK')
    return loaded


# Uncomment to load from a custom path:
# ovr_models = load_binary_models(model_dir='path/to/saved/models')

# Load all 5 from default OVR_MODEL_DIR:
ovr_models = load_binary_models()

## Agreement analysis

```
0 votes  -> no classifier is confident          -> disagreement (ambiguous)
1 vote   -> exactly one says yes                -> agreement
2+ votes -> multiple classifiers say yes        -> disagreement (conflict)
```

In [ ]:
@torch.no_grad()
def predict_binary_probs(models_dict, df, transform=VAL_TF):
    """
    Run all binary classifiers on df.
    Returns (N, 5) float32 array of positive-class probabilities.
    """
    ds     = InferenceDataset(df, transform)
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)
    N     = len(ds)
    probs = np.zeros((N, NUM_CLASSES), dtype=np.float32)

    for cid in sorted(models_dict.keys()):
        model = models_dict[cid]
        model.eval()
        preds_i = []
        for bi, (imgs, _) in enumerate(loader):
            p = torch.sigmoid(model(imgs.to(DEVICE))).squeeze(1)
            preds_i.extend(p.cpu().tolist())
            bar = '\u2588'*int(20*(bi+1)/len(loader)) + '\u2591'*(20-int(20*(bi+1)/len(loader)))
            print(f'\r  class {cid} |{bar}| {bi+1}/{len(loader)}', end='', flush=True)
        print()
        probs[:, cid] = preds_i

    return probs


def analyze_agreement(probs, true_labels=None, threshold=OVR_THRESHOLD):
    """
    Compute and display agreement statistics.
    If true_labels provided, also computes F1 scores per strategy.
    Returns a stats dict with masks and predictions.
    """
    votes   = (probs > threshold).astype(int)  # (N, 5)
    n_votes = votes.sum(axis=1)                 # (N,)

    mask_none     = n_votes == 0
    mask_agree    = n_votes == 1
    mask_conflict = n_votes >= 2
    mask_disagree = ~mask_agree

    argmax_preds = probs.argmax(axis=1)

    stats = {
        'total':         len(probs),
        'agreed':        mask_agree.sum(),
        'no_vote':       mask_none.sum(),
        'conflict':      mask_conflict.sum(),
        'disagreed':     mask_disagree.sum(),
        'mask_agree':    mask_agree,
        'mask_none':     mask_none,
        'mask_conflict': mask_conflict,
        'mask_disagree': mask_disagree,
        'argmax_preds':  argmax_preds,
        'votes':         votes,
        'n_votes':       n_votes,
    }

    total = stats['total']
    print(f'{"─"*55}')
    print(f'Agreement analysis  (threshold={threshold})')
    print(f'{"─"*55}')
    print(f'  Total instances:     {total:>6,}')
    print(f'  Agreement  (1 vote): {stats["agreed"]:>6,}  ({100*stats["agreed"]/total:.1f}%)')
    print(f'  No vote    (0):      {stats["no_vote"]:>6,}  ({100*stats["no_vote"]/total:.1f}%)')
    print(f'  Conflict   (>=2):    {stats["conflict"]:>6,}  ({100*stats["conflict"]/total:.1f}%)')
    print(f'  Disagreed total:     {stats["disagreed"]:>6,}  ({100*stats["disagreed"]/total:.1f}%)')

    print(f'\n  Vote distribution:')
    for nv, cnt in sorted(Counter(n_votes.tolist()).items()):
        bar = '\u2588' * int(25 * cnt / total)
        print(f'    {nv} votes: {cnt:>5,}  {bar}')

    if true_labels is not None:
        true_labels = np.asarray(true_labels)

        if mask_agree.any():
            agreed_preds = votes[mask_agree].argmax(axis=1)
            f1_agreed = f1_score(true_labels[mask_agree], agreed_preds,
                                 average='macro', zero_division=0)
        else:
            f1_agreed = float('nan')

        f1_argmax = f1_score(true_labels, argmax_preds, average='macro', zero_division=0)

        print(f'\n  Macro-F1 (agreed only):  {f1_agreed:.4f}')
        print(f'  Macro-F1 (argmax all):   {f1_argmax:.4f}')
        stats['f1_agreed'] = f1_agreed
        stats['f1_argmax'] = f1_argmax

    # ── Plots ────────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    vote_counts = [stats['agreed'], stats['no_vote'], stats['conflict']]
    vote_labels = ['Agreement\n(1 vote)', 'No vote\n(0)', 'Conflict\n(>=2)']
    colors      = ['#4caf50', '#ff9800', '#f44336']
    bars = axes[0].bar(vote_labels, vote_counts, color=colors, edgecolor='black', alpha=0.85)
    axes[0].set_title('OvR voting results'); axes[0].set_ylabel('Instances')
    for bar_obj, v in zip(bars, vote_counts):
        axes[0].text(bar_obj.get_x()+bar_obj.get_width()/2, bar_obj.get_height()+2,
                     f'{v:,}', ha='center', fontsize=10)

    axes[1].hist(n_votes, bins=np.arange(-0.5, NUM_CLASSES+1.5, 1),
                 edgecolor='black', alpha=0.75, color='steelblue')
    axes[1].axvline(1, color='green', linestyle='--', label='ideal (1 vote)')
    axes[1].set_xlabel('Number of "yes" votes'); axes[1].set_ylabel('Instances')
    axes[1].set_title('Vote distribution'); axes[1].legend()

    plt.tight_layout()
    plt.savefig('ovr_agreement.png', dpi=120, bbox_inches='tight')
    plt.show()

    return stats


print('Agreement utils OK')

In [ ]:
# Generate binary probabilities on the held-out val set
# (val_df was NOT seen during binary model training -> no data leakage)
print('Generating val probabilities for meta-classifier training...')
val_probs  = predict_binary_probs(ovr_models, val_df)
val_labels = val_df['class_id'].astype(int).values

val_stats = analyze_agreement(val_probs, true_labels=val_labels)

## Meta-classifier (tiebreaker)

Trained on the 5 binary probabilities from the held-out val set.  
Uses **Logistic Regression + isotonic calibration** so its output probabilities are well-calibrated.  

Three strategies are benchmarked:
| Strategy | Description |
|---|---|
| `argmax` | Always pick highest binary probability — no meta-clf needed |
| `meta` | Always use meta-classifier |
| `hybrid` | Agreed cases: OvR vote; disagreed: meta-classifier **(recommended)** |

In [ ]:
# ── Fit meta-classifier ───────────────────────────────────────────────────────
meta_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', CalibratedClassifierCV(
        LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
        ),
        method='isotonic',   # non-linear calibration; use 'sigmoid' if val set < 200
        cv=5,
    )),
])

meta_pipeline.fit(val_probs, val_labels)

meta_preds = meta_pipeline.predict(val_probs)
f1_meta    = f1_score(val_labels, meta_preds, average='macro', zero_division=0)

print(f'Meta-classifier (LogReg + isotonic calibration):')
print(f'  Macro-F1 on val: {f1_meta:.4f}')
print(f'\nClassification report:')
print(classification_report(val_labels, meta_preds,
                             target_names=list(CLASS_NAMES.values())))

# ── Benchmark three strategies ────────────────────────────────────────────────
f1_argmax = f1_score(val_labels, val_probs.argmax(axis=1),
                     average='macro', zero_division=0)

# Hybrid: agreed -> OvR, disagreed -> meta-clf
hybrid_preds = val_probs.argmax(axis=1).copy()
ma = val_stats['mask_agree']
md = val_stats['mask_disagree']
if ma.any():
    hybrid_preds[ma] = val_stats['votes'][ma].argmax(axis=1)
if md.any():
    hybrid_preds[md] = meta_pipeline.predict(val_probs[md])
f1_hybrid = f1_score(val_labels, hybrid_preds, average='macro', zero_division=0)

print('─'*55)
print('Strategy comparison on val:')
print(f'  argmax (no meta-clf): {f1_argmax:.4f}')
print(f'  meta always:          {f1_meta:.4f}')
print(f'  hybrid (recommended): {f1_hybrid:.4f}')
print('─'*55)

# ── Save meta-classifier ──────────────────────────────────────────────────────
with open(META_CLF_PATH, 'wb') as f:
    pickle.dump(meta_pipeline, f)
print(f'\nSaved meta-classifier -> {META_CLF_PATH}')

## Load meta-classifier from file

Run this cell to reload a previously saved meta-classifier without retraining.

In [ ]:
def load_meta_clf(path=META_CLF_PATH):
    with open(path, 'rb') as f:
        clf = pickle.load(f)
    print(f'Loaded meta-classifier <- {path}')
    return clf

# meta_pipeline = load_meta_clf()   # uncomment to load from file

## Full inference pipeline

In [ ]:
def predict_ovr(models_dict, df, meta_clf,
                threshold=OVR_THRESHOLD, strategy='hybrid'):
    """
    Full OvR inference.

    strategy:
        'hybrid'  — agreed: OvR vote; disagreed: meta-clf    (recommended)
        'meta'    — always use meta-clf
        'argmax'  — always argmax of binary probs (no meta-clf)

    Returns:
        preds  : (N,) int array of class predictions
        probs  : (N, 5) float array of binary probabilities
    """
    print(f'OvR inference  (strategy={strategy}, threshold={threshold})')
    probs = predict_binary_probs(models_dict, df)

    votes   = (probs > threshold).astype(int)
    n_votes = votes.sum(axis=1)
    mask_agree    = n_votes == 1
    mask_disagree = ~mask_agree

    preds = probs.argmax(axis=1).copy()

    if strategy == 'hybrid':
        if mask_agree.any():
            preds[mask_agree] = votes[mask_agree].argmax(axis=1)
        if mask_disagree.any():
            preds[mask_disagree] = meta_clf.predict(probs[mask_disagree])

    elif strategy == 'meta':
        preds = meta_clf.predict(probs)

    elif strategy == 'argmax':
        preds = probs.argmax(axis=1)

    n_total     = len(preds)
    n_agreed    = mask_agree.sum()
    n_disagreed = mask_disagree.sum()
    print(f'  Agreed    ({100*n_agreed/n_total:.1f}%): decided by OvR vote')
    print(f'  Disagreed ({100*n_disagreed/n_total:.1f}%): decided by meta-clf / argmax')

    return preds, probs


print('predict_ovr() OK')

## Generate submission

In [ ]:
test_preds, test_probs = predict_ovr(
    ovr_models, test, meta_pipeline, strategy='hybrid'
)

test_reset = test.reset_index(drop=True)
row_ids    = test_reset['row_id'].tolist()
submission = pd.DataFrame({'row_id': row_ids, 'class_id': test_preds})

sample_sub = pd.read_csv(BASE_DIR / 'sample_submission.csv')
assert set(submission['row_id']) == set(sample_sub['row_id']), 'row_id mismatch!'
assert submission['class_id'].between(0, 4).all(), 'Invalid class_id!'

submission = (
    submission.set_index('row_id')
              .reindex(sample_sub['row_id'])
              .reset_index()
)

print('Prediction distribution on test:')
for cid, cnt in sorted(submission['class_id'].value_counts().items()):
    bar = '\u2588' * int(25 * cnt / len(submission))
    print(f'  [{cid}] {CLASS_NAMES[cid]:25s}: {cnt:>5,}  ({100*cnt/len(submission):.1f}%)  {bar}')

submission.to_csv('submission_ovr.csv', index=False)
print('\nSaved -> submission_ovr.csv')

if IN_COLAB:
    from google.colab import files
    files.download('submission_ovr.csv')
    print('Download started.')

## Per-class probability heatmap (optional diagnostic)

In [ ]:
# Show average binary probabilities per true class on val set
# Diagonal should be bright; off-diagonal dark = good separation

heatmap = np.zeros((NUM_CLASSES, NUM_CLASSES))
for true_cls in range(NUM_CLASSES):
    mask = val_labels == true_cls
    if mask.any():
        heatmap[true_cls] = val_probs[mask].mean(axis=0)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(heatmap, vmin=0, vmax=1, cmap='RdYlGn')
plt.colorbar(im, ax=ax, label='Mean positive probability')
ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels([f'[{i}]\n{n.split("_")[0]}' for i, n in CLASS_NAMES.items()], fontsize=8)
ax.set_yticklabels([f'[{i}] {n}' for i, n in CLASS_NAMES.items()], fontsize=8)
ax.set_xlabel('Binary classifier ("is this class?")'); ax.set_ylabel('True class')
ax.set_title('OvR classifier responses per true class\n(diagonal = good, off-diagonal = confusion)')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax.text(j, i, f'{heatmap[i,j]:.2f}', ha='center', va='center', fontsize=9,
                color='black' if 0.3 < heatmap[i,j] < 0.7 else 'white')
plt.tight_layout()
plt.savefig('ovr_heatmap.png', dpi=130, bbox_inches='tight')
plt.show()